# Baseline Logistic Regression

- **Baseline Logistic Regression**
           - ↓ compare against
- **Weighted Logistic Regression**
- **XGBoost**
- **Isolation Forest**
- **Threshold tuning**

1. Define features and target
2. Split training and test data
3. Train Logistic Regression
4. Make predictions
5. Evaluate fraud detection performance

## Load dataset 


In [2]:
import pandas as pd

data_path = "../data/raw/creditcard.csv"
df = pd.read_csv(data_path)

In [4]:
X = df.drop("Class", axis=1)
y = df["Class"]

In [5]:
X.shape, y.shape

((284807, 30), (284807,))

## Split Data


In [8]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)# splititng data 80/20%

In [9]:
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((227845, 30), (56962, 30), (227845,), (56962,))

## feature scaling / standardization 


### Create the scaler

In [10]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

In [11]:
scaler.fit(X_train)

StandardScaler()

In [12]:
scaler.mean_

array([ 9.48850937e+04,  7.95290922e-04, -4.81013687e-04, -3.83022392e-04,
       -1.97223997e-05,  1.74584369e-04, -1.17118617e-03,  7.67387574e-05,
       -4.33928397e-04,  7.01756965e-04, -3.89440775e-04, -7.87247374e-04,
        2.71812789e-03, -4.93121683e-04, -5.86174977e-05,  7.44064525e-04,
       -1.06990340e-03,  3.90248279e-04,  5.62369121e-05,  6.63511187e-04,
       -9.98938638e-04,  3.69741873e-04,  3.02706709e-04,  5.03893413e-05,
       -2.77277157e-04, -6.55396231e-04,  1.26095357e-04, -7.04733971e-05,
        1.53721000e-04,  8.81762977e+01])

## Mean, Standard Deviation and variance ofe the Train data

In [13]:
import pandas as pd

scaler_stats = pd.DataFrame({
    "Feature": X_train.columns,
    "Mean": scaler.mean_,
    "Standard Deviation": scaler.scale_,
    "Variance": scaler.var_
})

scaler_stats

,Feature,Mean,Standard Deviation,Variance
0,Time,94885.093704,47488.310822,2.255140e+09
1,V1,0.000795,1.958921,3.837372e+00
2,V2,-0.000481,1.649089,2.719494e+00
3,V3,-0.000383,1.515289,2.296101e+00
4,V4,-0.000020,1.415864,2.004671e+00
5,V5,0.000175,1.379566,1.903203e+00
6,V6,-0.001171,1.331215,1.772133e+00
7,V7,0.000077,1.238428,1.533704e+00
8,V8,-0.000434,1.191479,1.419623e+00
9,V9,0.000702,1.097349,1.204175e+00


In [15]:
X_train[["Time", "Amount"]].describe()

,Time,Amount
count,227845.000000,227845.000000
mean,94885.093704,88.176298
std,47488.415034,250.722602
min,0.000000,0.000000
25%,54228.000000,5.640000
50%,84805.000000,22.000000
75%,139364.000000,77.490000
max,172792.000000,25691.160000


In [16]:
# Take one real Amount value from the training data
amount_value = X_train["Amount"].iloc[0]

# Get the learned mean and standard deviation for Amount
amount_mean = scaler.mean_[X_train.columns.get_loc("Amount")]
amount_std = scaler.scale_[X_train.columns.get_loc("Amount")]

# Apply the standardization formula manually
scaled_amount = (amount_value - amount_mean) / amount_std

print("Original Amount:", amount_value)
print("Training Mean:", amount_mean)
print("Training Standard Deviation:", amount_std)
print("Scaled Amount:", scaled_amount)

Original Amount: 7.32
Training Mean: 88.17629770238538
Training Standard Deviation: 250.72205133874502
Scaled Amount: -0.3224937625974599


In [17]:
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [18]:
type(X_train_scaled), X_train_scaled.shape

(numpy.ndarray, (227845, 30))

## Logistic Regression

In [19]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression()

## Train Model

In [20]:
model.fit(X_train_scaled, y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneOnly used for `solver` == 'sag', 'saga' or 'liblinear' to shuffle thedata. It has no effect on the other solvers.See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following

In [21]:
y_pred = model.predict(X_test_scaled)

In [22]:
y_pred[:20]

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0])

In [23]:
import numpy as np

np.unique(y_pred, return_counts=True)

(array([0, 1]), array([56887,    75]))

In [24]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred)
cm

array([[56851,    13],
       [   36,    62]])

In [25]:
from sklearn.metrics import precision_score

precision = precision_score(y_test, y_pred)
precision

0.8266666666666667

In [26]:
from sklearn.metrics import recall_score

recall = recall_score(y_test, y_pred)
recall

0.6326530612244898

In [27]:
from sklearn.metrics import f1_score

f1 = f1_score(y_test, y_pred)
f1

0.7167630057803468

In [28]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test, y_pred)
accuracy

0.9991397773954567

### PR-AUC.

In [30]:
y_prob = model.predict_proba(X_test_scaled)[:, 1]

In [31]:
y_prob

array([6.66589544e-05, 3.65855417e-05, 6.07869406e-06, ...,
       1.57444971e-05, 6.55799811e-05, 2.65939506e-04], shape=(56962,))

In [32]:
from sklearn.metrics import average_precision_score

pr_auc = average_precision_score(y_test, y_prob)
pr_auc

0.7413820992780461